In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


In [2]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

transform_test = transforms.Compose([
    
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

full_train_data = datasets.CIFAR10(root="../data", train = True, download=True, transform=transform_test)
train_data = datasets.CIFAR10(root="../data", train = True, download=True, transform=transform_train)
test_data = datasets.CIFAR10(root="../data", train = False, download=True, transform=transform_test)

train_size = int(len(train_data) * .8)
val_size = len(train_data) - train_size

train_subset, val_subset = torch.utils.data.random_split(full_train_data, [train_size, val_size], generator= torch.Generator().manual_seed(123))

train_data = Subset(train_data, train_subset.indices)
val_data = Subset(full_train_data, val_subset.indices)

In [3]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

torch.Size([64, 3, 32, 32])
torch.Size([64])


In [4]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

cuda


In [5]:
class patch_embedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=128):  #in channels is 3 for RGB images
        super().__init__()
        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x) #(64, 128, 8, 8) -> (batch_size, embed_dim, num_patches_h, num_patches_w)
        x = x.flatten(2) #flatten the last two dimensions (H and W)
        x = x.transpose(1, 2) #transpose to (batch_size, num_patches, embed_dim)
        # 64 patches of size 128
        return x


In [6]:
class SmallViT(nn.Module):
    def __init__(self,img_size=32, patch_size=4, in_channels=3, embed_dim=128, depth=4, num_heads=4, mlp_dim=512, num_classes=10, droupout=0.1):
        super().__init__()
        self.patch_embed = patch_embedding(img_size=img_size, patch_size=patch_size, in_channels=in_channels, embed_dim=embed_dim) 
        #what each patfch looks like after being projected to the embedding dimension

        num_patches = (img_size // patch_size) ** 2 # should be 64 for 32x32 images with 4x4 patches

        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, embed_dim)) #learnable positional embeddings, nn.Parameter allows us to learn these during training
        self.droupout = nn.Dropout(.1) #dropout for regularization
        #where each patch is in the image

        #everything so for is for a transformer to adapt to an image input, now we need to add the transformer encoder layers

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, #token size, number of features in each patch embedding
            nhead=num_heads,  
            dim_feedforward=mlp_dim,
            dropout=droupout,
            activation="gelu", #counterpart to relu, smoother and better for training deep networks and transformers
            batch_first=True   #we want our input to be (batch_size, seq_len, embed_dim)
        )

        self.encoder = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=depth) #repeat the encoder layer 'depth' times
        #encoder layer consists of multi-head self attention and a feedforward network, we stack 'depth' of these layers to create the full transformer encoder

        self.norm = nn.LayerNorm(embed_dim) #layer normalization for stability and better training, normalizes the output of the encoder
        self.head = nn.Linear(embed_dim, num_classes) #final linear layer to get class logits(10 classes for CIFAR-10)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed
        x = self.droupout(x)
        x = self.encoder(x)
        x = self.norm(x)  #layer norm for stability and better training, normalizes the output of the encoder
        x = x.mean(dim=1) #global average pooling, we take the mean of the patch embeddings to get a single vector representation of the image
        x= self.head(x)   #final linear layer to get class logits(10 classes for CIFAR-10)
        return x
    

model = SmallViT().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
print(model)

SmallViT(
  (patch_embed): patch_embedding(
    (proj): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
  )
  (droupout): Dropout(p=0.1, inplace=False)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=128, out_features=10, bias=True)
)


In [7]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy



In [8]:
import copy
epochs = 50
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
best_val_loss = float("inf")
best_state = None

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"LR: {current_lr:.6f} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())

    scheduler.step()


Epoch 1/50 | Train Loss: 1.8075 | Val Loss: 1.5692 | Val Acc: 0.4217
Epoch 2/50 | Train Loss: 1.5055 | Val Loss: 1.4112 | Val Acc: 0.4781
Epoch 3/50 | Train Loss: 1.3851 | Val Loss: 1.3336 | Val Acc: 0.5185
Epoch 4/50 | Train Loss: 1.3067 | Val Loss: 1.2339 | Val Acc: 0.5548
Epoch 5/50 | Train Loss: 1.2481 | Val Loss: 1.2127 | Val Acc: 0.5564
Epoch 6/50 | Train Loss: 1.1975 | Val Loss: 1.1542 | Val Acc: 0.5909
Epoch 7/50 | Train Loss: 1.1571 | Val Loss: 1.1254 | Val Acc: 0.5974
Epoch 8/50 | Train Loss: 1.1235 | Val Loss: 1.0820 | Val Acc: 0.6115
Epoch 9/50 | Train Loss: 1.0837 | Val Loss: 1.0906 | Val Acc: 0.6072
Epoch 10/50 | Train Loss: 1.0555 | Val Loss: 1.0573 | Val Acc: 0.6222
Epoch 11/50 | Train Loss: 1.0281 | Val Loss: 1.0118 | Val Acc: 0.6402
Epoch 12/50 | Train Loss: 0.9950 | Val Loss: 0.9984 | Val Acc: 0.6475
Epoch 13/50 | Train Loss: 0.9728 | Val Loss: 0.9789 | Val Acc: 0.6517
Epoch 14/50 | Train Loss: 0.9508 | Val Loss: 0.9454 | Val Acc: 0.6630
Epoch 15/50 | Train Loss: 0.9

In [9]:
model.load_state_dict(best_state)

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Test Loss: 0.7884 | Test Acc: 0.7515
